# atari_1m — DQN on Pong (4M frames, default GPU kernels)

One plot: the `dqn_pong` component's 5 seeds as faint `tab:blue` curves, with
the across-seed mean as a thicker `tab:blue` curve. Atari is cluster-scale, so this
notebook expects results synced from the cluster (`run.py sync`).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

_HERE = Path.cwd()
_EXP_DIR = _HERE if (_HERE / "config.py").exists() else Path("experiments/atari_1m")
sys.path.insert(0, str(_EXP_DIR))
sys.path.insert(0, str(_EXP_DIR.resolve().parents[1]))

from experiment import load_result, load_runs
from analysis.plotting import mean_over_seeds, seed_grids_for, style

from config import EXPERIMENT

FONT_SIZE = 20   # 2x matplotlib's default (10), passed to every text call below

COMPONENT = "dqn_pong"


def grid_for(component, n=500):
    """A shared [0, total_steps] timestep grid from any run's length."""
    df = load_runs(EXPERIMENT, component)
    total_steps = len(load_result(EXPERIMENT, component, df["run_id"][0])["reward"])
    return np.linspace(0, total_steps, n)


def plot_seeds_and_mean(ax, grid, stack, label, color):
    """Draw every seed as a faint curve plus the across-seed mean, thicker."""
    for row in stack:
        ax.plot(grid, row, lw=1, color=color, alpha=0.15)
    mean = mean_over_seeds(stack)
    m = ~np.isnan(mean)
    ax.plot(grid[m], mean[m], lw=1.75, color=color, label=label)


In [ ]:
HERE = Path.cwd()
RESULTS = HERE / "results" if (HERE / "results").exists() else _EXP_DIR / "results"

GRID = grid_for(COMPONENT)

# GRID is in env steps; scale to frames (steps x frameskip) for the x-axis.
FRAMESKIP = load_runs(EXPERIMENT, COMPONENT)["ENV_HYPERS.FRAMESKIP"][0]
FRAME_GRID = GRID * FRAMESKIP
FRAME_TICKS = np.array([1, 2, 3, 4]) * 1_000_000
FRAME_TICK_LABELS = [f"{t // 1_000_000}M" for t in FRAME_TICKS]


In [ ]:
stack = seed_grids_for(EXPERIMENT, COMPONENT, GRID)

fig, ax = plt.subplots(figsize=(9, 6))   # 2:3 height:width
plot_seeds_and_mean(ax, FRAME_GRID, stack, "DQN", "tab:blue")
ax.set_title("Pong", fontsize=FONT_SIZE)
ax.legend(loc="upper left", frameon=False, fontsize=FONT_SIZE)
style(ax, xlabel="Frames")
ax.set_ylabel(
    "Return", rotation=90, ha="center", va="center",
    labelpad=20, fontsize=FONT_SIZE,
)
ax.set_xlabel(ax.get_xlabel(), fontsize=FONT_SIZE)
ax.set_xticks(FRAME_TICKS)
ax.set_xticklabels(FRAME_TICK_LABELS)
ax.tick_params(labelsize=FONT_SIZE)
fig.tight_layout()
plt.show()


In [ ]:
PLOTS_DIR = RESULTS.parent / "plots"
PLOTS_DIR.mkdir(exist_ok=True)
fig.savefig(PLOTS_DIR / f"{COMPONENT}.pdf", bbox_inches="tight")
print(f"saved 1 plot to {PLOTS_DIR}")
